# Project 9 — Local Study Notes Generator

## Turn Text into Notes, Flashcards, and Quizzes

**Goal:** Transform lesson content into structured study materials:
outline notes, flashcards, and self-test questions.

**Stack:** Ollama · LangChain · Jupyter

### What You'll Learn
1. Multi-output generation from one source
2. Structured output formatting
3. Difficulty-graded question generation

### Prerequisites
- **Ollama** with a chat model (e.g., `qwen3:8b`)

In [ ]:
!pip install -q langchain langchain-ollama

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = r.json().get("models", [])
    print(f"Ollama running — {len(models)} model(s) available")
    for m in models[:5]: print(f"  - {m['name']}")
except Exception as e:
    print(f"Cannot reach Ollama at {OLLAMA_BASE}: {e}")
    print("Start Ollama first: ollama serve")

## Step 2 — Sample Lesson Content

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

lesson = """Neural Networks Fundamentals

A neural network is a computational model inspired by the human brain.
It consists of layers of interconnected nodes (neurons). Each connection
has a weight that adjusts during training.

Key components:
- Input layer: receives raw data
- Hidden layers: learn intermediate representations
- Output layer: produces predictions
- Activation functions: introduce non-linearity (ReLU, sigmoid, tanh)
- Loss function: measures prediction error
- Backpropagation: algorithm to update weights using gradient descent

Training process:
1. Forward pass: data flows through the network
2. Loss calculation: compare output to target
3. Backward pass: compute gradients
4. Weight update: adjust weights to reduce loss
5. Repeat for many epochs

Common architectures: feedforward, CNN (images), RNN (sequences),
Transformer (attention-based). Overfitting is prevented via dropout,
early stopping, and regularization."""
print(f"Lesson: {len(lesson)} chars")

## Step 3 — Outline Notes

In [ ]:
notes_prompt = ChatPromptTemplate.from_messages([
    ("system", "Create structured study notes from this lesson. "
     "Use hierarchical format with main topics, subtopics, and key details. "
     "Keep it concise but complete."),
    ("human", "{lesson}")
])
notes = (notes_prompt | llm | StrOutputParser()).invoke({"lesson": lesson})
print("📝 STUDY NOTES:\n")
print(notes)

## Step 4 — Flashcards

In [ ]:
flash_prompt = ChatPromptTemplate.from_messages([
    ("system", "Create 8 flashcards from this lesson. "
     "Format each as: Q: [question]\nA: [answer]\n"
     "Cover key concepts, definitions, and processes."),
    ("human", "{lesson}")
])
flashcards = (flash_prompt | llm | StrOutputParser()).invoke({"lesson": lesson})
print("🃏 FLASHCARDS:\n")
print(flashcards)

## Step 5 — Self-Test Questions

In [ ]:
quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate a self-test with 3 difficulty levels:\n"
     "- 3 EASY questions (definitions/recall)\n"
     "- 3 MEDIUM questions (understanding/application)\n"
     "- 2 HARD questions (analysis/synthesis)\n"
     "Include answers after all questions."),
    ("human", "{lesson}")
])
quiz = (quiz_prompt | llm | StrOutputParser()).invoke({"lesson": lesson})
print("🧠 SELF-TEST:\n")
print(quiz)

## Step 6 — Summary Card

In [ ]:
card_prompt = ChatPromptTemplate.from_messages([
    ("system", "Create a one-page summary card: "
     "3 key concepts, 3 formulas/rules, 3 common mistakes to avoid."),
    ("human", "{lesson}")
])
card = (card_prompt | llm | StrOutputParser()).invoke({"lesson": lesson})
print("🎯 SUMMARY CARD:\n")
print(card)

## Limitations

1. **Quality depends on model** — smaller models may miss nuances.
2. **No adaptive difficulty** — doesn't adjust to learner level.
3. **No spaced repetition** — no scheduling for review.

## What You Learned

| Concept | What We Did |
|---|---|
| **Outline notes** | Structured topic hierarchy |
| **Flashcards** | Q&A pairs for memorization |
| **Self-test** | Multi-difficulty questions |
| **Summary card** | One-page quick reference |